# 🧬 Connect AI — 장기 기억 학습 (Unsloth)
내 1인 기업 지식을 모델 **가중치에 체득**시킵니다. 위 메뉴 **런타임 → 모두 실행**만 누르면 됩니다 (무료 T4 GPU).
- 데이터셋: `catmind/catmind_dataset` (단기 지식 → conversations Q&A)
- 베이스 모델: `unsloth/gemma-4-E2B-it`  ← *내가 쓰는 모델로 바꿔도 됩니다 (누적 학습)*
- 결과 모델: `catmind/Marketing-ai-DX-v11` (GGUF — Connect AI 내장 엔진에 바로 로드, LM Studio 불필요)
- 설정: rank 16/alpha 32 · dropout 0 · lr 0.0003 · steps 40 · seq 1024 · linear · 양자화 q4_k_m (데이터 18개)


In [4]:
%%capture
# 버전을 직접 고정하지 않는다 — Unsloth가 현재 Colab torch에 맞는 의존성(torchao·transformers 등)을 알아서 설치.
# (고정 레시피는 Colab torch가 바뀌면 register_constant/recompile_limit 같은 충돌이 연쇄로 난다)
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo


## 🔑 HuggingFace 로그인 (맨 먼저!)
아래 칸에 **write 토큰**을 붙여넣으세요. *비공개 데이터셋을 불러오고*, 학습된 모델을 *업로드*하는 데 둘 다 필요해요.


In [5]:
from huggingface_hub import notebook_login
notebook_login()


In [7]:
# 🔐 로그인을 맨 앞에서 확인 — 안 돼 있으면 긴 학습 전에 바로 멈춰서 시간 낭비 방지
from huggingface_hub import HfApi
try:
    print("✅ 로그인됨:", HfApi().whoami()["name"], "— 결과는 내 계정에 올라가요")
except Exception:
    raise SystemExit("❌ 먼저 위 🔑 칸에 HuggingFace write 토큰을 붙여넣고 Login을 누르세요. 그다음 [런타임 → 모두 실행]을 다시 누르면 됩니다.")


✅ 로그인됨: catmind — 결과는 내 계정에 올라가요


In [8]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None, max_seq_length = 1024,
    load_in_4bit = True, full_finetuning = False,
)
print("✅ 베이스 모델 로딩 완료")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/301k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/16.8k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

✅ 베이스 모델 로딩 완료


In [9]:
# LoRA — 전체의 1% 미만만 학습(메모리↓, 페르소나·핵심지식엔 충분)
model = FastModel.get_peft_model(
    model, finetune_language_layers=True, finetune_attention_modules=True,
    finetune_mlp_modules=True, finetune_vision_layers=False,
    r = 16, lora_alpha = 32, lora_dropout = 0, bias = "none", random_state = 3407,
)


## 📦 단기 지식 데이터셋 (conversations Q&A)
Connect AI 앱이 업로드한 데이터셋. 각 행 = `{conversations:[{user},{assistant}]}`


In [10]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
ds = load_dataset("catmind/catmind_dataset", data_files="connect-ai-brain.jsonl", split="train", token=True)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
def fmt(ex):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix("<bos>") for c in ex["conversations"]]
    return {"text": texts}
ds = ds.map(fmt, batched=True)
print("데이터 개수:", len(ds)); print(ds[0]["text"][:400])


connect-ai-brain.jsonl:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

데이터 개수: 18
<|turn>user
제품 자체를 '보랏빛 소'로 만들려면 마케팅 전략을 세우기 전에 어떤 점들을 가장 먼저 고민해야 할까요?<turn|>
<|turn>model
# 보랏빛 소가 온다 (Purple Cow)
# 보랏빛 소가 온다 (Purple Cow) — 마케팅 완전 정리

세스 고딘(Seth Godin)의 마케팅 고전. 한 문장: "평범하면 보이지 않는다. 주목할 만하게(remarkable) 만들어라."

## 1. 보랏빛 소 = 리마커블
- 평범한 소 수백 마리는 안 보인다. 보랏빛 소는 누구나 멈춰서 보고 친구에게 말한다.
- 리마커블 = "remark(언급)할 만한". 마케팅의 핵심은 광고 기술이 아니라 제품 자체가 주목할 만한가다.
- 리마커블의 반대말은 "나쁨"이 아니라 "매우 좋음(very go


In [11]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 40, learning_rate = 0.0003,
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.001,
        lr_scheduler_type = "linear", seed = 3407, report_to = "none",
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/18 [00:00<?, ? examples/s]

In [12]:
# 🎭 응답(assistant)만 학습 — 질문 패턴은 마스킹(효율↑·품질↑)
# ⚠️ 마커는 계열마다 다름 → 실제 텍스트에서 자동 감지 (gemma·llama·qwen 모두 지원)
from unsloth.chat_templates import train_on_responses_only
_t = ds[0]["text"]
if "<start_of_turn>user" in _t: _im, _rm = "<start_of_turn>user\n", "<start_of_turn>model\n"
elif "<|start_header_id|>" in _t: _im, _rm = "<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"
elif "<|im_start|>" in _t: _im, _rm = "<|im_start|>user\n", "<|im_start|>assistant\n"
elif "<|turn>user" in _t: _im, _rm = "<|turn>user\n", "<|turn>model\n"
else: _im, _rm = None, None
if _rm:
    trainer = train_on_responses_only(trainer, instruction_part=_im, response_part=_rm)
    print(f"✅ 마스킹 마커 자동감지: {_rm.strip()} — 응답만 학습")
else:
    print("ℹ️ 마커 자동감지 실패 → 전체 텍스트로 학습(문제 없음)")


Map (num_proc=6):   0%|          | 0/18 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/18 [00:00<?, ? examples/s]

✅ 마스킹 마커 자동감지: <|turn>model — 응답만 학습


In [13]:
trainer_stats = trainer.train()
print("🎉 학습 완료! 최종 loss:", round(trainer_stats.training_loss, 4))
print("💡 loss 0.2~0.4면 sweet spot. 너무 낮으면(<0.1) 과적합 — max_steps 줄이세요.")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18 | Num Epochs = 8 | Total steps = 40
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 25,337,856 of 5,148,515,872 (0.49% trained)


Step,Training Loss
1,0.927210
2,1.450786
3,1.012408
4,1.184399
5,2.914918
6,1.036275
7,0.999301
8,1.022615
9,0.564618
10,1.288560


🎉 학습 완료! 최종 loss: 0.4779
💡 loss 0.2~0.4면 sweet spot. 너무 낮으면(<0.1) 과적합 — max_steps 줄이세요.


## 🧪 학습된 모델 테스트 (업로드 전에 확인!)
내가 가르친 지식을 직접 물어보세요. 답에 그 내용이 나오면 학습 성공이에요. 질문은 자유롭게 바꿔도 됩니다.


In [14]:
from unsloth import FastModel
FastModel.for_inference(model)
def chat(prompt, max_tokens=220):
    try:
        msg = [{"role":"user","content":[{"type":"text","text":prompt}]}]
        inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    except Exception:
        msg = [{"role":"user","content":prompt}]
        inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    inp = inp.to(model.device)
    if inp["input_ids"][0,0].item() == tokenizer.bos_token_id:
        inp["input_ids"] = inp["input_ids"][:,1:]; inp["attention_mask"] = inp["attention_mask"][:,1:]
    out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\u2753 {prompt}\n\U0001F4AC {ans}\n" + "\u2500"*58)

# 👇 내가 가르친 지식에 대해 물어보세요 (자유롭게 수정)
chat("내 사업/지식에 대해 아는 걸 알려줘")
chat
"너는 무엇을 도와줄 수 있어?"



❓ 내 사업/지식에 대해 아는 걸 알려줘
💬 [운영 2026-06-23 사이클#49] 내 사업/지식에 대해 아는 걸 알려줘
──────────────────────────────────────────────────────────
❓ 너는 무엇을 도와줄 수 있어?
💬 저는 Google에서 개발한 대규모 언어 모델인 'Gemma 4'입니다.
[운영 2026-06-22 사이클#29]
저는 다양한 언어를 이해하고 생성할 수 있으며, 다음과 같은 일을 도와드릴 수 있습니다.

**#글쓰기 도우미**
*   보고서, 기사 초안 작성
*   이메일, 자기소개서, 후기 자동 생성
*   운영 2026-06-22 사이클#29: 영숙: 지난달 인기 유튜브 영상 분석 후 다음 기획안 초안 작성 (오늘업무_2026-06-22/보고서.md)

**#지식 검색기**
*   다양한 주제에 대한 사실 확인 ("지구 온난화의 주요 원인은?")
*   복잡한 개념을 쉽게 설명 ("양자 얽힘을 초등학생에게 설명해 줘.")
*   실시간 뉴스 요
──────────────────────────────────────────────────────────


## 💾 저장 → HuggingFace
**safetensors(AI 진화·합성용) + GGUF(앱 실행용)** 둘 다 올라가요. (맨 앞에서 로그인했으니 바로 됩니다)


In [ ]:
# 메모리 정리(OOM 방지) — 학습기 메모리 해제 후 변환
import gc, torch
try:
    del trainer
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()
# 📤 저장 위치 = "내" HF 계정 (위에서 로그인한 본인 계정으로 자동 — 노트북이 공유돼도 안전)
from huggingface_hub import HfApi
NAME = "Marketing-ai-DX-v11"
OUTPUT = f'{HfApi().whoami()["name"]}/{NAME}'
print("📤 내 계정에 저장:", OUTPUT)
# ① 합성용 safetensors (AI 진화소에서 다시 합칠 수 있어요 — 이게 없으면 합성 불가!)
try:
    model.push_to_hub_merged(OUTPUT, tokenizer, save_method="merged_16bit", token=True)
    print("✅ safetensors 업로드 — AI 진화소에서 합치기 가능")
except Exception as e:
    print("⚠️ 병합 업로드 실패 → 어댑터(LoRA)로 폴백:", e)
    model.push_to_hub(OUTPUT, token=True); tokenizer.push_to_hub(OUTPUT, token=True)
# ② 앱 실행용 GGUF
model.push_to_hub_gguf(OUTPUT, tokenizer, quantization_method="q4_k_m", token=True)
print(f"✅ 완료! safetensors(합성용)+GGUF(실행용) 둘 다 → Connect AI 앱 🤖 내 AI 에서 {OUTPUT} 받기")
